# 05 — GRACE Diagnostics

Reproduces paper **Fig. 10** (`GRACE_all_method_effects.png`),
**Fig. 11** (`cluster_hallucinating.png`), **Fig. 12**
(`gg_centric_rep_fragmentation.png`).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as scipy_stats

from grace.analysis.load_results import load_summary_results
from grace.diagnostics.alignment import alignment_per_layer
from grace.diagnostics.fragmentation import pl_ra_correlation
from grace.diagnostics.pair_heatmap import per_pair_similarity

FIG_DIR = Path('Images'); FIG_DIR.mkdir(exist_ok=True)
MODELS = {'gemma2': 'google/gemma-2-2b-it', 'gemma3': 'google/gemma-3-27b-it',
          'llama3': 'meta-llama/Llama-3.3-70B-Instruct'}
MODEL_SHORT = {'gemma2': 'Gemma2-2B', 'gemma3': 'Gemma3-27B', 'llama3': 'Llama-70B'}
MODEL_COLORS = {
    'gemma2': '#e74c3c',
    'gemma3': '#3498db',
    'llama3': '#2ecc71',
}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

## Fig. 10 — Per-(model, concept) Δsteerability vs. PV baseline for the 3 GRACE methods

For each (model, concept), look up the best-utility configuration for `pv`,
`unit_mean`, and `cluster`. Plot the per-method delta as a heatmap.


In [ ]:
def best_utility(model_name, concept, method):
    rows = [r for r in load_summary_results(model_name, concept, method=method)
            if r.get('mean_utility') is not None]
    return max((r['mean_utility'] for r in rows), default=None)

# Build delta tables per model
model_order = ['gemma2', 'gemma3', 'llama3']
delta_cols = ['cluster', 'unit_mean']
sub_labels = ['Cluster', 'Unit-mean']

delta_tables = {}
for tag in model_order:
    mname = MODELS[tag]
    rows = []
    for c in CONCEPTS:
        pv = best_utility(mname, c, 'pv')
        if pv is None:
            rows.append({'concept': c, 'cluster': np.nan, 'unit_mean': np.nan})
            continue
        row = {'concept': c}
        for m in delta_cols:
            v = best_utility(mname, c, m)
            row[m] = (v - pv) if v is not None else np.nan
        rows.append(row)
    delta_tables[tag] = pd.DataFrame(rows).set_index('concept')

# Build (20 x 6) matrix: 2 methods x 3 models
blocks = []
col_labels = []
for tag in model_order:
    dt = delta_tables[tag]
    block = dt[delta_cols].reindex(CONCEPTS)
    blocks.append(block.values)
    for sl in sub_labels:
        col_labels.append(sl)
data = np.column_stack(blocks)

vmax = np.nanmax(np.abs(data))
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(np.nan_to_num(data, nan=0), cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

# Annotate cells
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        if np.isnan(val):
            txt = '--'
            color = 'gray'
        else:
            txt = f'{val:.1f}'
            color = 'white' if abs(val) > 0.55 * vmax else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=7, color=color)

# Vertical separators between model groups
n_methods = len(delta_cols)
for sep in range(1, len(model_order)):
    ax.axvline(sep * n_methods - 0.5, color='black', linewidth=1.5)

# Model group labels above
for g, tag in enumerate(model_order):
    ax.text(g * n_methods + (n_methods - 1) / 2, -1.6, MODEL_SHORT[tag],
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(len(CONCEPTS)))
ax.set_yticklabels([c.replace('_', ' ') for c in CONCEPTS], fontsize=8)
ax.set_xlim(-0.5, len(col_labels) - 0.5)
ax.set_ylim(len(CONCEPTS) - 0.5, -0.5)

cbar = fig.colorbar(im, ax=ax, shrink=0.75, pad=0.02)
cbar.set_label('Steerability delta vs PV baseline', fontsize=9)

plt.tight_layout()
fig.savefig(FIG_DIR / 'GRACE_all_method_effects.png', dpi=300, bbox_inches='tight')
plt.show()

## Fig. 11 — Cluster diagnostic for `hallucinating`


In [ ]:
def plot_pair_sim_heatmap(ax, sims_dict, title, highlight_pairs=None):
    """Plot 5x5 cosine-similarity heatmap from per_pair_similarity output."""
    if not sims_dict:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
        return
    avg_sim = np.mean(np.stack([s.numpy() for s in sims_dict.values()]), axis=0)
    n = avg_sim.shape[0]
    im = ax.imshow(avg_sim, cmap='RdBu_r', vmin=-1, vmax=1)

    # Annotate cells
    for i in range(n):
        for j in range(n):
            color = 'white' if abs(avg_sim[i, j]) > 0.6 else 'black'
            ax.text(j, i, f'{avg_sim[i, j]:.2f}', ha='center', va='center',
                    fontsize=8, color=color)

    # Red borders around highlighted pairs
    if highlight_pairs:
        for p in highlight_pairs:
            ax.add_patch(Rectangle((-0.5, p - 0.5), n, 1,
                                   fill=False, edgecolor='red', linewidth=2.5))
            ax.add_patch(Rectangle((p - 0.5, -0.5), 1, n,
                                   fill=False, edgecolor='red', linewidth=2.5))

    ax.set_xticks(range(n))
    ax.set_xticklabels([f'Pair {i}' for i in range(n)], fontsize=8)
    ax.set_yticks(range(n))
    ax.set_yticklabels([f'Pair {i}' for i in range(n)], fontsize=8)
    ax.set_title(title, fontsize=10)
    return im

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel 1: Gemma 2 2B similarity heatmap
sims_2b = per_pair_similarity(MODELS['gemma2'], 'hallucinating', 'response_avg')
plot_pair_sim_heatmap(axes[0], sims_2b,
                      'Gemma 2 2B: pairs 3, 4 misaligned',
                      highlight_pairs=[3, 4])

# Panel 2: steerability vs coefficient comparison
ax = axes[1]
for method, color, marker in [('pv', 'steelblue', 'o'), ('cluster', '#e67e22', 's')]:
    rows = [r for r in load_summary_results(MODELS['gemma2'], 'hallucinating', method=method)
            if r.get('mean_utility') is not None]
    if not rows:
        continue
    mdf = pd.DataFrame(rows).groupby('coef')['mean_utility'].max().reset_index()
    ax.plot(mdf.coef, mdf.mean_utility, f'{marker}-', label=method.upper(),
            color=color, markersize=4, linewidth=1.5)
ax.set_xlabel('Steering coefficient', fontsize=9)
ax.set_ylabel('Steerability', fontsize=9)
ax.set_title('Gemma 2 2B: cluster rescues high-coef regime', fontsize=10)
ax.legend(fontsize=8)

# Panel 3: Gemma 3 27B similarity heatmap
sims_27b = per_pair_similarity(MODELS['gemma3'], 'hallucinating', 'response_avg')
plot_pair_sim_heatmap(axes[2], sims_27b,
                      'Gemma 3 27B: pairs 1, 4 misaligned',
                      highlight_pairs=[1, 4])

plt.tight_layout()
fig.savefig(FIG_DIR / 'cluster_hallucinating.png', dpi=300, bbox_inches='tight')
plt.show()

## Fig. 12 — Representational fragmentation diagnostic for `golden_gate_centric`

Left: PL/RA correlation across all (model, concept) pairs vs. constrained-search delta.
Right: layer profiles for `golden_gate_centric` on Gemma-2-2B (low correlation case).


In [ ]:
rows = []
for tag, mname in MODELS.items():
    for c in CONCEPTS:
        try:
            r, A_pl, A_ra = pl_ra_correlation(mname, c)
        except FileNotFoundError:
            continue
        pv_unc = best_utility(mname, c, 'pv')
        if pv_unc is None:
            continue
        # Compute delta for top-15 PL constraint
        all_layers = sorted(set(A_pl) & set(A_ra))
        top15_pl = set(sorted(all_layers, key=lambda l: A_pl.get(l, 0), reverse=True)[:15])
        # Get grid results to compute delta
        grid_rows = load_summary_results(mname, c, method='pv')
        if not grid_rows:
            continue
        gdf = pd.DataFrame(grid_rows).dropna(subset=['mean_utility'])
        best_overall = gdf['mean_utility'].max()
        constrained = gdf[gdf['layer'].isin(top15_pl)]
        best_constrained = constrained['mean_utility'].max() if len(constrained) > 0 else np.nan
        d_pl = best_constrained - best_overall if not np.isnan(best_constrained) else np.nan
        rows.append({'model': tag, 'concept': c, 'act_type_corr': r,
                     'd_top15_pl': d_pl, 'pv_unc': pv_unc})
fdf = pd.DataFrame(rows)

# --- Plot ---
fig = plt.figure(figsize=(12, 5))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 2], hspace=0.5)

# Panel 1: scatter (spans both rows on the left)
ax = fig.add_subplot(gs[:, 0])
for tag in ['gemma2', 'gemma3', 'llama3']:
    sub = fdf[fdf['model'] == tag]
    ax.scatter(sub['act_type_corr'], sub['d_top15_pl'], c=MODEL_COLORS[tag],
               label=MODEL_SHORT[tag], alpha=0.7, s=50, edgecolors='white', linewidth=0.5)

# Regression line
valid = fdf[['act_type_corr', 'd_top15_pl']].dropna()
if len(valid) > 2:
    slope, intercept, r, p, se = scipy_stats.linregress(valid['act_type_corr'], valid['d_top15_pl'])
    x_range = np.linspace(valid['act_type_corr'].min(), valid['act_type_corr'].max(), 50)
    ax.plot(x_range, intercept + slope * x_range, 'k--', alpha=0.5, linewidth=1.5)

# golden_gate_centric annotation
gg = fdf[(fdf['model'] == 'gemma2') & (fdf['concept'] == 'golden_gate_centric')]
if not gg.empty:
    ax.annotate('golden_gate_centric', xy=(gg.iloc[0]['act_type_corr'], gg.iloc[0]['d_top15_pl']),
                xytext=(0.15, -8), fontsize=8,
                arrowprops=dict(arrowstyle='->', color='black', lw=0.8))

ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(0.2, color='red', linestyle='--', alpha=0.5, label='correlation = 0.2')
ax.set_xlabel('Activation-type correlation (prompt-last vs response-avg)', fontsize=9)
ax.set_ylabel('Delta steerability (top-15 PL vs unconstrained)', fontsize=9)
ax.set_title(f'Activation-type correlation predicts failure\nr = {r:.3f}, p = {p:.4f}',
             fontsize=10)
ax.legend(fontsize=7, loc='lower right')

# Panel 2a (top-right): PL alignment profile
ax2a = fig.add_subplot(gs[0, 1])
try:
    _, A_pl, A_ra = pl_ra_correlation(MODELS['gemma2'], 'golden_gate_centric')
    layers = sorted(set(A_pl) & set(A_ra))
    ax2a.plot(layers, [A_pl[l] for l in layers], 'ko-', markersize=3, linewidth=1.2,
              label='PL alignment')
    ax2a.plot(layers, [A_ra[l] for l in layers], 's-', color='#e67e22', markersize=3,
              linewidth=1.2, label='RA alignment')
    ax2a.set_xlabel('Layer', fontsize=8)
    ax2a.set_ylabel('avg_cosine_sim', fontsize=8)
    ax2a.set_title('golden_gate_centric / Gemma 2 2B\nPL vs RA profiles', fontsize=9)
    ax2a.legend(fontsize=7)
except FileNotFoundError:
    ax2a.text(0.5, 0.5, 'No data', transform=ax2a.transAxes, ha='center')

# Panel 2b (bottom-right): strategy comparison bar chart
ax2b = fig.add_subplot(gs[1, 1])
# Compute union strategy delta
union_deltas = []
pl_deltas = []
for _, row in fdf.iterrows():
    pl_deltas.append(row['d_top15_pl'])
    # Union not available from frag data alone; use PL as placeholder
    union_deltas.append(row['d_top15_pl'])

strategies = ['d_top15_pl']
strat_labels = ['Top-15 PL']
strat_colors = ['#e74c3c']
means = [fdf['d_top15_pl'].mean()]
n_failures = [(fdf['d_top15_pl'] < -1).sum()]

ax2b.bar(range(len(strategies)), means, color=strat_colors, edgecolor='black',
         linewidth=0.5, width=0.55)
ax2b.axhline(0, color='gray', linestyle=':', alpha=0.5)
for i, m in enumerate(means):
    va = 'top' if m < 0 else 'bottom'
    offset = -0.04 if m < 0 else 0.04
    ax2b.text(i, m + offset, f'{m:.2f}', ha='center', va=va, fontsize=10, fontweight='bold')
ax2b.set_xticks(range(len(strategies)))
ax2b.set_xticklabels(strat_labels, fontsize=9)
ax2b.set_ylabel('Mean delta steerability', fontsize=9)
ax2b.set_title(f'Constraint cost: {n_failures[0]} failures (delta < -1)', fontsize=10)

plt.tight_layout()
fig.savefig(FIG_DIR / 'gg_centric_rep_fragmentation.png', dpi=300, bbox_inches='tight')
plt.show()